# RAG — Voice of Customer com Olist

Notebook principal da entrega. Executa a jornada completa sobre as tabelas reais do Olist: validação dos dados, enriquecimento relacional, construção do corpus, embeddings, FAISS, recuperação semântica, geração fundamentada e testes de cenários sem evidência.


## 0. Ambiente e imports

Se `python-dotenv`, `faiss-cpu`, `sentence-transformers` ou `openai` não estiverem instalados, execute a instalação abaixo no mesmo kernel selecionado pelo VS Code. Depois reinicie o kernel.


In [1]:
import sys
from pathlib import Path
REQ_FILE = Path.cwd() / "requirements.txt"
if not REQ_FILE.exists():
    REQ_FILE = Path.cwd().parent / "requirements.txt"
print("Requirements:", REQ_FILE)
!{sys.executable} -m pip install -r "{REQ_FILE}"


Requirements: f:\FIAP\TechChallenge-Fase4\requirements.txt



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# Localiza a raiz do projeto, inclusive quando o notebook é aberto pela pasta notebooks.
ROOT = Path.cwd().resolve()

if (ROOT / "config" / "settings.py").exists():
    pass
elif ROOT.name == "notebooks" and (ROOT.parent / "config" / "settings.py").exists():
    ROOT = ROOT.parent
else:
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "config" / "settings.py").exists():
            ROOT = parent
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Carrega automaticamente o .env localizado na raiz do projeto.
ENV_FILE = ROOT / ".env"
if ENV_FILE.exists():
    load_dotenv(ENV_FILE, override=False)
    print(f"✓ .env carregado: {ENV_FILE}")
else:
    print(f"⚠ .env não encontrado: {ENV_FILE}")

from config.settings import settings
from src.data.loader import load_olist_raw
from src.data.preprocessing import enrich_reviews, build_documents
from src.embeddings.embedder import EmbeddingModel
from src.retrieval.retriever import VectorRetriever
from src.rag.pipeline import RAGPipeline
from src.generation.llm import LLMGenerator

RAW_DIR = settings.data_dir / "raw"
INDEX_DIR = settings.data_dir / "vectorstore"
print("Projeto:", ROOT)
print("RAW:", RAW_DIR)


✓ .env carregado: F:\FIAP\TechChallenge-Fase4\.env
Projeto: F:\FIAP\TechChallenge-Fase4
RAW: F:\FIAP\TechChallenge-Fase4\data\raw


## 1. Validação das tabelas RAW

O corpus textual é formado pelas avaliações. As demais tabelas enriquecem cada avaliação por `order_id`, `customer_id`, `product_id` e `seller_id`.


In [3]:
expected = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
status = pd.DataFrame({"arquivo": expected, "existe": [(RAW_DIR / f).exists() for f in expected]})
display(status)
missing = status.loc[~status["existe"], "arquivo"].tolist()
if missing:
    raise FileNotFoundError("Arquivos ausentes em data/raw: " + ", ".join(missing))


,arquivo,existe
0,olist_customers_dataset.csv,True
1,olist_geolocation_dataset.csv,True
2,olist_order_items_dataset.csv,True
3,olist_order_payments_dataset.csv,True
4,olist_order_reviews_dataset.csv,True
5,olist_orders_dataset.csv,True
6,olist_products_dataset.csv,True
7,olist_sellers_dataset.csv,True
8,product_category_name_translation.csv,True


In [4]:
raw = load_olist_raw(RAW_DIR)
summary = pd.DataFrame([{"tabela": k, "linhas": len(v), "colunas": len(v.columns)} for k, v in raw.items()])
display(summary)


,tabela,linhas,colunas
0,reviews,99224,7
1,orders,99441,8
2,items,112650,7
3,customers,99441,5
4,products,32951,9
5,sellers,3095,4
6,payments,103886,5
7,categories,71,2


## 2. Construção da base de conhecimento

Uma avaliação permanece uma única unidade semântica. Itens, categorias, vendedores e pagamentos são agregados por pedido antes do enriquecimento, evitando duplicação de comentários no índice.


In [5]:
reviews = raw["reviews"]
print("Reviews totais:", len(reviews))
print("Review IDs distintos:", reviews["review_id"].nunique())
print("Comentários preenchidos:", reviews["review_comment_message"].fillna("").astype(str).str.strip().ne("").sum())

prepared = enrich_reviews(raw)
documents = build_documents(prepared)
print("Documentos finais:", len(documents))
print("Exemplo de documento:")
display(pd.DataFrame([documents[0]]))


Reviews totais: 99224
Review IDs distintos: 98410
Comentários preenchidos: 40950
Documentos finais: 40950
Exemplo de documento:


,document_id,text,metadata
0,e64fb393e7b32834bb789ff8bb30750e,Recebi bem antes do prazo estipulado.,{'review_id': 'e64fb393e7b32834bb789ff8bb30750...


In [6]:
metadata_df = pd.DataFrame([d["metadata"] for d in documents])
metadata_cols = [c for c in ["review_score","customer_state","product_category_name_english","delivery_days","delivery_delay_days","payment_type"] if c in metadata_df.columns]
display(metadata_df[metadata_cols].head())


,review_score,customer_state,product_category_name_english,delivery_days,delivery_delay_days,payment_type
0,5,SC,garden_tools,10.64400462962963,-19.619039351851853,credit_card
1,5,SC,sports_leisure,18.232314814814814,-8.310011574074075,credit_card
2,4,SP,small_appliances,7.307754629629629,-15.255416666666667,credit_card
3,4,SP,health_beauty,3.1314814814814813,-10.196597222222222,credit_card
4,5,SP,computers_accessories,3.8597453703703706,-15.375844907407407,credit_card


## 3. Embeddings e índice vetorial

Modelo: `paraphrase-multilingual-MiniLM-L12-v2`. Os vetores são normalizados e indexados no FAISS com similaridade por produto interno, equivalente a cosine similarity após normalização.


In [7]:
embedder = EmbeddingModel(settings.embedding_model)
texts = [d["text"] for d in documents]
embeddings = embedder.encode(texts)
retriever = VectorRetriever(embeddings, documents)
retriever.save(INDEX_DIR)
print("Modelo:", settings.embedding_model)
print("Dimensão:", embeddings.shape[1])
print("Vetores indexados:", retriever.size)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/640 [00:00<?, ?it/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dimensão: 384
Vetores indexados: 40950


## 4. Recuperação semântica


In [8]:
question = "Quais são os principais problemas relatados pelos clientes sobre a entrega?"
evidence = retriever.search(embedder.encode([question])[0], top_k=settings.top_k, min_score=settings.min_relevance_score)
print("Pergunta:", question)
print("Evidências recuperadas:", len(evidence))
for item in evidence:
    print(f"[{item['document_id']}] score={item['score']:.3f} | nota={item.get('metadata',{}).get('review_score')} | {item['text']}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Pergunta: Quais são os principais problemas relatados pelos clientes sobre a entrega?
Evidências recuperadas: 5
[42a3a3e2b79465364e24f6f4c642bc23] score=0.734 | nota=1 | Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto entregue, porém foi entregue apenas 1 produto, aguardo solução!
[e06d9a717c61331208eabed007431c41] score=0.719 | nota=4 | Embora tenha havido problemas com a entrega pelos correios o produto chegou.
[949eca021ed60d00a2d2bda3cc686d4d] score=0.714 | nota=1 | muitos desencontros nas entregas de produtos
[7d8d252cc0ae002b13db033ec1cd9dcb] score=0.699 | nota=1 | Demora de entrega
[a553875413fd30a7f78cd701b0064a6f] score=0.693 | nota=5 | Recomendo o produto e o fornecedor, pois o atraso na entrega foi causado pelos serviços dos Correios.


### Configuração do LLM e dos guardrails

O projeto usa **Ollama + Llama 3.2** por padrão. A geração acontece localmente em `http://localhost:11434`, portanto não exige créditos de API.

A validação ocorre em duas etapas antes da geração:
- `MIN_RELEVANCE_SCORE`: elimina candidatos muito fracos no retrieval;
- `MIN_SCOPE_SCORE`: exige relevância mínima do melhor candidato;
- `MIN_DOMAIN_SCORE`: exige afinidade semântica da pergunta com âncoras curadas do domínio Olist.

Assim, vizinhos retornados pelo FAISS não são automaticamente tratados como evidência.

In [9]:
print("LLM_PROVIDER:", settings.llm_provider)
print("LLM_MODEL:", settings.llm_model)
print("MIN_RELEVANCE_SCORE:", settings.min_relevance_score)
print("MIN_SCOPE_SCORE:", settings.min_scope_score)
print("MIN_DOMAIN_SCORE:", settings.min_domain_score)
if settings.llm_provider == "ollama":
    print("✓ LLM gratuito/local selecionado: Ollama.")
else:
    print("✓ Provider externo selecionado:", settings.llm_provider)

LLM_PROVIDER: ollama
LLM_MODEL: llama3.2
MIN_RELEVANCE_SCORE: 0.25
MIN_SCOPE_SCORE: 0.55
MIN_DOMAIN_SCORE: 0.45
✓ LLM gratuito/local selecionado: Ollama.


## 5. Geração RAG fundamentada

A geração recebe **somente as evidências consideradas suficientes** pelo pipeline. O prompt instrui o modelo a não inventar fatos, declarar insuficiência quando necessário e citar os documentos utilizados.

Um ponto importante da arquitetura é distinguir **candidatos recuperados** de **evidências aceitas**. O FAISS sempre encontra vizinhos; isso não significa que a pergunta possa ser respondida pela base.


In [10]:
pipeline = None
try:
    generator = LLMGenerator(
        model=settings.llm_model,
        provider=settings.llm_provider,
        base_url=settings.ollama_base_url if settings.llm_provider == "ollama" else None,
    )
    pipeline = RAGPipeline(
        embedder,
        retriever,
        generator,
        settings.top_k,
        settings.min_relevance_score,
        settings.min_scope_score,
        settings.min_domain_score,
    )
    result = pipeline.ask(question)
    print(result["answer"])
    print("\nStatus da evidência:", result["evidence_status"])
    print("Score de domínio:", round(result["domain_score"], 4))
    print("Score do melhor candidato:", round(result["top_retrieval_score"], 4) if result["top_retrieval_score"] is not None else None)
    print("IDs das evidências:", [x["document_id"] for x in result["evidence"]])
except Exception as exc:
    print("⚠ A geração LLM não foi executada.")
    print("Detalhe:", exc)
    print("Confirme se o Ollama está em execução e se o modelo existe: ollama list")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Com base nas evidências recuperadas, é possível identificar alguns problemas relatados pelos clientes sobre a entrega:

1. **Divergências na entrega**: Alguns clientes relatam que o produto entregue não é o mesmo que o que foi pedido, como mencionado no ID [42a3a3e2b79465364e24f6f4c642bc23]. Isso sugere que há um problema com a verificação da ordem de entrega ou com a entrega de apenas um produto.
2. **Problemas com os correios**: Alguns clientes relatam problemas com a entrega devido aos serviços dos correios, como mencionado no ID [e06d9a717c61331208eabed007431c41] e [a553875413fd30a7f78cd701b0064a6f]. Isso sugere que há um problema com a entrega devido às condições dos correios.
3. **Atrasos na entrega**: Alguns clientes relatam atrasos na entrega, como mencionado no ID [7d8d252cc0ae002b13db033ec1cd9dcb]. Isso sugere que há um problema com a velocidade da entrega.

É importante notar que nem todos os clientes relatam problemas com a entrega, e alguns até relatam que o produto chegou

## 6. Cenários de evidência

São demonstrados separadamente três comportamentos: consulta respondível, ausência de evidência e pergunta fora do escopo. Isso evita considerar automaticamente qualquer resultado do FAISS como evidência.


In [11]:
if pipeline is not None:
    supported_question = "Quais são os principais problemas relacionados à entrega?"
    result_supported = pipeline.ask(supported_question)
    print("CONSULTA NO DOMÍNIO")
    print(result_supported["answer"])
    print("Status:", result_supported["evidence_status"])
    print("Score de domínio:", round(result_supported["domain_score"], 4))
    print("Melhor score FAISS:", round(result_supported["top_retrieval_score"], 4))
    print("Evidências aceitas:", len(result_supported["evidence"]))

    out_of_scope = "Qual é a previsão do tempo para amanhã?"
    result_out = pipeline.ask(out_of_scope)
    print("\nCONSULTA FORA DO ESCOPO")
    print(result_out["answer"])
    print("Status:", result_out["evidence_status"])
    print("Score de domínio:", round(result_out["domain_score"], 4))
    print("Melhor score FAISS:", round(result_out["top_retrieval_score"], 4) if result_out["top_retrieval_score"] is not None else None)
    print("Candidatos recuperados pelo FAISS:", len(result_out["retrieved_candidates"]))
    print("Evidências aceitas:", len(result_out["evidence"]))
    print("Tem evidência:", result_out["has_evidence"])
else:
    print("Pipeline indisponível porque o LLM local não está acessível. A etapa de recuperação continua reproduzível.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

CONSULTA NO DOMÍNIO
Com base nas evidências recuperadas da base Olist, os principais problemas relacionados à entrega identificados pelos clientes são:

Demora de entrega e Atraso na entrega.

As evidências recuperadas mostram que os clientes mencionaram demora de entrega e atraso na entrega em várias ocasiões. Por exemplo, o cliente com o ID [048f89d5550f6d495dbfb726b23ba2a1] mencionou atraso na entrega em 2018-04-13, enquanto o cliente com o ID [a1074ef6728469383f2a8bf58fd98e51] também mencionou atraso na entrega em 2018-02-24. Além disso, os clientes com os IDs [db6e80117789aa15d03c3406c7429845] e [b02ae085148e5e1eb6b11142ea9c653c] mencionaram demora na entrega em 2017-04-21 e 2017-10-31, respectivamente.

É importante notar que não há evidências suficientes para identificar outros problemas relacionados à entrega, como perda de pacotes ou entrega errada.
Status: supported
Score de domínio: 0.8949
Melhor score FAISS: 0.7659
Evidências aceitas: 5


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


CONSULTA FORA DO ESCOPO
A pergunta parece estar fora do escopo da base de conhecimento Olist ou não possui evidências suficientes relacionadas ao tema. Não é possível responder com segurança usando as avaliações disponíveis.
Status: out_of_scope
Score de domínio: 0.0654
Melhor score FAISS: 0.6198
Candidatos recuperados pelo FAISS: 5
Evidências aceitas: 0
Tem evidência: False


## 7. Exemplos de consultas sobre a Voz do Cliente

Os exemplos abaixo representam perguntas dentro do domínio. O notebook mostra o status da evidência para deixar explícito que a resposta depende de recuperação válida.


In [ ]:
questions = [
    "Quais são os principais problemas relacionados à entrega?",
    "Quais reclamações aparecem nas avaliações de baixa nota?",
    "O que os clientes relatam sobre a qualidade dos produtos?",
]

if pipeline is not None:
    for q in questions:
        result = pipeline.ask(q)
        print("\nPERGUNTA:", q)
        print("STATUS:", result["evidence_status"])
        print("SCORE DOMÍNIO:", round(result["domain_score"], 4))
        print("EVIDÊNCIAS:", len(result["evidence"]))
        print("RESPOSTA:", result["answer"][:800])

## 8. Persistência e reprodução

O índice produzido pelo notebook é salvo em `data/vectorstore/`. A mesma construção pode ser reproduzida pelo script `python scripts/build_index.py`. Para consultas no terminal: `python scripts/query_rag.py "sua pergunta"`. Para a interface: `streamlit run app/streamlit_app.py`.


In [13]:
loaded = VectorRetriever.load(INDEX_DIR)
print("Índice persistido e recarregado com sucesso.")
print("Vetores:", loaded.size)
print("Documentos persistidos:", len(loaded.documents))
assert loaded.size == len(documents)


Índice persistido e recarregado com sucesso.
Vetores: 40950
Documentos persistidos: 40950


## 9. Avaliação quantitativa do retrieval

O desafio exige recuperar conteúdos relevantes. Para tornar essa verificação mensurável, o projeto mantém um conjunto anotado em `data/evaluation/retrieval_gold.json` e calcula Precision@K, Recall@K e MRR sobre o retrieval bruto.

> O conjunto é uma primeira referência anotada a partir das evidências observadas na demonstração executada. Ele deve ser ampliado quando novas perguntas forem avaliadas manualmente.


In [ ]:
import json
from src.evaluation.evaluator import evaluate_queries

GOLD_FILE = ROOT / "data" / "evaluation" / "retrieval_gold.json"
with GOLD_FILE.open(encoding="utf-8") as f:
    retrieval_gold = json.load(f)

retrieval_rows = evaluate_queries(pipeline, retrieval_gold, k=settings.top_k)
retrieval_eval = pd.DataFrame(retrieval_rows)
display(retrieval_eval[["question", "precision_at_k", "recall_at_k", "mrr"]])

summary = pd.DataFrame([{
    f"Precision@{settings.top_k}": retrieval_eval["precision_at_k"].mean(),
    f"Recall@{settings.top_k}": retrieval_eval["recall_at_k"].mean(),
    "MRR": retrieval_eval["mrr"].mean(),
}])
display(summary)


## 10. Contexto efetivamente enviado ao LLM

A solução separa **candidatos recuperados** de **evidências aceitas**. Abaixo o contexto é exibido explicitamente antes da geração, permitindo auditar quais avaliações sustentam a resposta.


In [ ]:
context_demo = pipeline.ask("Quais são os principais problemas relacionados à entrega?")
print("Status:", context_demo["evidence_status"])
print("Candidatos FAISS:", len(context_demo["retrieved_candidates"]))
print("Evidências aceitas:", len(context_demo["evidence"]))
print("\n===== CONTEXTO ENVIADO AO LLM =====\n")
print(context_demo["context"])


## 11. Cenário de evidência insuficiente

Além do caso fora do escopo, o pipeline deve saber operar quando nenhum candidato ultrapassa o limiar mínimo de relevância. O teste abaixo usa deliberadamente um limiar muito alto para validar esse caminho de segurança.


In [ ]:
strict_pipeline = RAGPipeline(
    embedder,
    retriever,
    generator,
    top_k=settings.top_k,
    min_relevance_score=0.99,
    min_scope_score=settings.min_scope_score,
    min_domain_score=settings.min_domain_score,
)
result_insufficient = strict_pipeline.ask("Quais são os principais problemas relacionados à entrega?")
print("Status:", result_insufficient["evidence_status"])
print("Candidatos:", len(result_insufficient["retrieved_candidates"]))
print("Evidências aceitas:", len(result_insufficient["evidence"]))
print("Tem evidência:", result_insufficient["has_evidence"])
print("Resposta:", result_insufficient["answer"])
assert result_insufficient["evidence_status"] == "insufficient_evidence"
assert result_insufficient["has_evidence"] is False


## 12. Checklist final de conformidade

- Base de conhecimento construída e enriquecida.
- Textos limpos e valores ausentes tratados.
- Uma avaliação mantida como unidade semântica.
- Embeddings e índice vetorial persistidos.
- Retrieval semântico implementado.
- Contexto do LLM explicitamente auditável.
- Respostas fundamentadas e rastreáveis por `document_id`.
- Ausência de evidência tratada.
- Pergunta fora do escopo tratada.
- Avaliação quantitativa de retrieval incorporada.
- Exemplo de pergunta corrigido para não confundir categoria de produto com qualidade do produto.


## 9. Conclusão técnica

O fluxo implementado é: **pergunta → embedding → recuperação semântica → validação de domínio e suficiência → contexto de evidências → LLM → resposta fundamentada**.

A validação diferencia candidatos recuperados pelo FAISS de evidências efetivamente aceitas. Para aceitar uma consulta, o pipeline exige simultaneamente relevância do melhor candidato e afinidade semântica com âncoras do domínio Olist.

Cada documento mantém `document_id` e metadados para rastreabilidade. A arquitetura diferencia três situações importantes: evidência suficiente, ausência de evidência e pergunta fora do escopo.

Essa distinção é relevante porque um mecanismo vetorial sempre pode retornar os vizinhos mais próximos disponíveis. O sistema não deve interpretar essa recuperação automática como prova de que a pergunta é respondível.

O notebook demonstra a jornada acadêmica completa, enquanto os módulos em `src/` concentram a implementação reutilizável para CLI e Streamlit.